# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (locked this week): Lane 4 — CTR / Engagement Opportunity Scoring.** Confirmed, not switched.

**My rule, in plain words:** a visible page is worth reviewing if it gets real impressions but its
click-through rate sits well below the typical CTR for its search-position tier — it is *shown a lot
but under-clicked*, which usually points at a weak title or meta description. This is the transparent
baseline my Week-5 model must beat.


In [6]:
# ---- Setup: same warehouse connection as ML-04 ----
import duckdb, os, json, numpy as np, pandas as pd
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

BASE   = "hf://datasets/FlyRank/internship-warehouse"
MONTH  = "2026-03"
FACT_M = f"{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIMC   = f"{BASE}/dim_content.parquet"
FLOOR  = 500
os.makedirs("work/outputs", exist_ok=True)

# Build the page-month slice (visible pages only), with position tier + expected CTR + shortfall.
df = con.sql(f'''
  WITH agg AS (
    SELECT content_hash_id,
           ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions)      AS impressions_m,
           SUM(gsc_clicks)           AS clicks_m,
           SUM(gsc_sum_position)*1.0/NULLIF(SUM(gsc_impressions),0) AS avg_position_m
    FROM read_parquet("{FACT_M}")
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= {FLOOR}
  )
  SELECT a.*,
         a.clicks_m*100.0/NULLIF(a.impressions_m,0) AS ctr,
         d.content_type, d.main_intent, d.word_count,
         CASE WHEN a.avg_position_m<=3 THEN 'top_3'
              WHEN a.avg_position_m<=10 THEN 'page_1'
              WHEN a.avg_position_m<=20 THEN 'striking'
              WHEN a.avg_position_m<=50 THEN 'page_3_5'
              ELSE 'deep' END AS position_tier
  FROM agg a JOIN read_parquet("{DIMC}") d USING (content_hash_id)
''').df()
print("Visible pages in scope:", len(df), "| median CTR:", round(df["ctr"].median(),3), "%")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Visible pages in scope: 61924 | median CTR: 0.178 %


## 1. Check two signals first, then state the rule

*Two bucket tables (with n). At least one signal is behind a real FlyRank flag from the session.*

Both signals below sit behind real FlyRank flags: **CTR-vs-position** is the logic behind the
"needs CTR fix" flag, and **impressions volume** is the logic behind the "quick win" flag. If either
signal is not real in my slice, my rule is built on sand — so I check before I encode.


In [7]:
# SIGNAL 1 (flag-linked: needs-CTR-fix) -- does CTR really fall as position gets worse?
order = {"top_3":0,"page_1":1,"striking":2,"page_3_5":3,"deep":4}
s1 = (df.groupby("position_tier")
        .agg(n=("ctr","size"), median_ctr=("ctr","median"))
        .reset_index())
s1["o"]=s1["position_tier"].map(order); s1=s1.sort_values("o").drop(columns="o")
print("SIGNAL 1 — CTR by position tier (n printed):")
print(s1.round(3).to_string(index=False))
mono = s1["median_ctr"].is_monotonic_decreasing
print("Verdict:", "CONFIRMED" if mono else "MIXED",
      "-- CTR", "falls" if mono else "does not cleanly fall", "as position worsens.\n")

# SIGNAL 2 (flag-linked: quick-win) -- is there real volume sitting at low CTR to win back?
df["vol_bucket"] = pd.cut(df["impressions_m"],
    bins=[500,1000,3000,10000,1e9],
    labels=["500-1k","1k-3k","3k-10k","10k+"], right=False)
s2 = (df.groupby("vol_bucket", observed=True)
        .agg(n=("ctr","size"), median_ctr=("ctr","median"))
        .reset_index())
print("SIGNAL 2 — CTR by impressions volume (n printed):")
print(s2.round(3).to_string(index=False))
lo, hi = s2["median_ctr"].iloc[0], s2["median_ctr"].iloc[-1]
rising = s2["median_ctr"].iloc[:-1].is_monotonic_increasing
if rising and hi > lo:
    print("Verdict: OPPOSITE -- I expected high-volume pages to sit at LOW CTR (the naive quick-win")
    print("premise). The data says the reverse: median CTR RISES with volume",
          f"({lo:.3f}% -> {hi:.3f}%).")
    print("This is a win, because it just saved my rule. If I had scored pages on volume alone I would")
    print("have flagged the pages that are already performing best. So volume must NOT be a reason to")
    print("flag a page on its own -- it is only a weight on a shortfall that is measured WITHIN a")
    print("position tier. My rule below does exactly that.")
else:
    print("Verdict: MIXED -- no clean monotonic relationship between volume and CTR.")


SIGNAL 1 — CTR by position tier (n printed):
position_tier     n  median_ctr
        top_3  7940       0.236
       page_1 32300       0.215
     striking 10534       0.167
     page_3_5 10468       0.082
         deep   682       0.000
Verdict: CONFIRMED -- CTR falls as position worsens.

SIGNAL 2 — CTR by impressions volume (n printed):
vol_bucket     n  median_ctr
    500-1k 16866       0.142
     1k-3k 22901       0.177
    3k-10k 16280       0.219
      10k+  5877       0.200
Verdict: OPPOSITE -- I expected high-volume pages to sit at LOW CTR (the naive quick-win
premise). The data says the reverse: median CTR RISES with volume (0.142% -> 0.200%).
This is a win, because it just saved my rule. If I had scored pages on volume alone I would
have flagged the pages that are already performing best. So volume must NOT be a reason to
flag a page on its own -- it is only a weight on a shortfall that is measured WITHIN a
position tier. My rule below does exactly that.


## 2. Encode ONE rule: score, reason code, action — write the ranked queue

*A transparent score (no fitted weights), one reason code, one action label.*

Expected CTR for a page = the median CTR of its own position tier. The **shortfall** is how far the
page sits below that expectation. The score weights the shortfall by impressions, so a big gap on a
high-traffic page ranks first. Every scored page gets the reason code `low_ctr_visible` and the
action `review title & meta`. No fitted weights, no future window — a rule a human can read.


In [8]:
# Expected CTR = median CTR of the page's position tier (transparent, no fitted weights).
tier_expected = df.groupby("position_tier")["ctr"].transform("median")
df["expected_ctr"]  = tier_expected
df["ctr_shortfall"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)
df["score"]         = df["ctr_shortfall"] * df["impressions_m"]
df["reason_code"]   = "low_ctr_visible"
df["action"]        = "review title & meta"

queue = (df[df["score"]>0]
         .sort_values("score", ascending=False)
         .loc[:, ["content_hash_id","position_tier","impressions_m","ctr",
                  "expected_ctr","ctr_shortfall","score","reason_code","action"]]
         .reset_index(drop=True))
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Ranked queue written to work/outputs/baseline_action_score.csv  |  rows:", len(queue))
print(queue.head(10).round(3).to_string(index=False))

# Commit-worthy receipts: a metrics JSON (the CSV itself stays out of git).
metrics = {
  "month": MONTH, "visible_pages": int(len(df)),
  "scored_pages": int((df["score"]>0).sum()),
  "median_ctr_pct": round(float(df["ctr"].median()),3),
  "top10_mean_impressions": round(float(queue.head(10)["impressions_m"].mean()),1),
  "top10_mean_shortfall_pp": round(float(queue.head(10)["ctr_shortfall"].mean()),3),
  "top10_zero_click_share": round(float((queue.head(10)["ctr"]==0).mean()),2),
}
with open("work/outputs/baseline_metrics.json","w") as f: json.dump(metrics,f,indent=2)
print("\nReceipts -> work/outputs/baseline_metrics.json"); print(metrics)


Ranked queue written to work/outputs/baseline_action_score.csv  |  rows: 30616
         content_hash_id position_tier  impressions_m   ctr  expected_ctr  ctr_shortfall     score     reason_code              action
content_44f34c0a90047651         top_3       212404.0 0.011         0.236          0.225 47714.990 low_ctr_visible review title & meta
content_8e1334d6356668e3         top_3       134984.0 0.001         0.236          0.235 31748.373 low_ctr_visible review title & meta
content_fec55986a1868d62         top_3       124075.0 0.001         0.236          0.235 29174.483 low_ctr_visible review title & meta
content_34a70fea29d15f24        page_1       143019.0 0.030         0.215          0.185 26390.773 low_ctr_visible review title & meta
content_f6116743b00afc2d        page_1       107584.0 0.014         0.215          0.201 21586.695 low_ctr_visible review title & meta
content_7c6373141eae744a        page_1       132593.0 0.063         0.215          0.152 20153.433 low_ctr_visi

## 3. Top-10 review — one line each: action, why, what would make it wrong

*Read my own top ten with a skeptic's eye. The "what would make it wrong" line is the point.*


In [9]:
top10 = queue.head(10).merge(df[["content_hash_id","main_intent","clicks_m"]], on="content_hash_id", how="left")
NEAR_ZERO = 0.02   # CTR at or below this is effectively a zero-click page
def wrong_note(r):
    notes = []
    if r["ctr"] <= NEAR_ZERO:
        notes.append(f"this is effectively a ZERO-CLICK page ({int(r['clicks_m'])} clicks on "
                     f"{int(r['impressions_m'])} impressions) -- wrong if the answer is shown on the "
                     f"results page itself, or if these impressions come from a query the page was "
                     f"never meant to win")
    if r["position_tier"] == "top_3":
        notes.append("it already ranks in the top 3, so wrong if the traffic is branded/navigational "
                     "where a low CTR is normal and a rewrite cannot help")
    if str(r["main_intent"]).lower() == "navigational":
        notes.append("navigational intent -- a low CTR is expected here, not a title fault")
    if r["impressions_m"] > 150000:
        notes.append("very high impression count -- wrong if one huge low-intent query is dominating "
                     "and dragging the whole page average down")
    if not notes:
        notes.append("wrong if the low CTR is an intent mismatch (right title, wrong audience) rather "
                     "than a weak title/meta")
    return "; ".join(notes)

print("TOP-10 REVIEW\n" + "="*70)
for i, r in top10.iterrows():
    print(f"#{i+1}  {r['action']}  |  tier={r['position_tier']}, impr={int(r['impressions_m'])}, "
          f"CTR={r['ctr']:.2f}% vs expected {r['expected_ctr']:.2f}%")
    print(f"     WHY: shown a lot but under-clicked — CTR is {r['ctr_shortfall']:.2f}pp below its tier's normal.")
    print(f"     WRONG IF: {wrong_note(r)}")


TOP-10 REVIEW
#1  review title & meta  |  tier=top_3, impr=212404, CTR=0.01% vs expected 0.24%
     WHY: shown a lot but under-clicked — CTR is 0.22pp below its tier's normal.
     WRONG IF: this is effectively a ZERO-CLICK page (24 clicks on 212404 impressions) -- wrong if the answer is shown on the results page itself, or if these impressions come from a query the page was never meant to win; it already ranks in the top 3, so wrong if the traffic is branded/navigational where a low CTR is normal and a rewrite cannot help; very high impression count -- wrong if one huge low-intent query is dominating and dragging the whole page average down
#2  review title & meta  |  tier=top_3, impr=134984, CTR=0.00% vs expected 0.24%
     WHY: shown a lot but under-clicked — CTR is 0.24pp below its tier's normal.
     WRONG IF: this is effectively a ZERO-CLICK page (1 clicks on 134984 impressions) -- wrong if the answer is shown on the results page itself, or if these impressions come from a query 

## 4. Weak picks (where my own rule is shakiest)

*The top-10 review is honest only if it names its weakest flags.*


In [10]:
NEAR_ZERO = 0.02
weak = top10[(top10["ctr"] <= NEAR_ZERO) | (top10["main_intent"].str.lower()=="navigational")]
print(f"{len(weak)} of my top 10 are shaky picks -- effectively zero-click (CTR <= {NEAR_ZERO}%) or navigational.")
print(f"That is {len(weak)} of the 10 rows my rule is most confident about.")
print("")
print("These are exactly where the rule is most likely wrong: a zero-click page may be answered on")
print("the SERP, and a navigational/brand page can carry a low CTR for reasons a title rewrite won't")
print("fix. A Week-5 model that also sees intent and content type should down-rank these — that is")
print("the improvement I will measure the model against this frozen baseline.")


7 of my top 10 are shaky picks -- effectively zero-click (CTR <= 0.02%) or navigational.
That is 7 of the 10 rows my rule is most confident about.

These are exactly where the rule is most likely wrong: a zero-click page may be answered on
the SERP, and a navigational/brand page can carry a low CTR for reasons a title rewrite won't
fix. A Week-5 model that also sees intent and content type should down-rank these — that is
the improvement I will measure the model against this frozen baseline.


## Self-check

- [x] Two signal checks with visible bucket tables and n; at least one behind a real FlyRank flag (CTR-vs-position and volume)
- [x] One transparent rule: a score, one reason code (`low_ctr_visible`), one action (`review title & meta`)
- [x] Ranked queue written from the notebook to work/outputs/baseline_action_score.csv (stays out of git)
- [x] Metrics receipts committed: work/outputs/baseline_metrics.json
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived model inputs; the rule reads observed metrics only, and stays FROZEN for the model comparison
- [x] Lane confirmed: Lane 4 (CTR / Engagement Opportunity Scoring)
